***
<center>IMPORTS</center>


In [80]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import altair as alt

***
<center> READ FILE<center>

In [43]:
file = pd.read_excel('Инфляция и ключевая ставка Банка России_F17_09_2013_T17_11_2026.xlsx', dtype = {'Дата': str})
file = file.rename(columns = {'Дата': 'date', 'Ключевая ставка, % годовых': 'key_rate', 'Инфляция, % г/г': 'infl', 'Цель по инфляции': 'target'})
file['date'] = pd.to_datetime(file['date'], format = '%m.%Y')

C:\Users\marr0\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [44]:
file.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      157 non-null    datetime64[ns]
 1   key_rate  157 non-null    float64       
 2   infl      157 non-null    float64       
 3   target    117 non-null    float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 5.0 KB


In [45]:
file.head()

,date,key_rate,infl,target
0,2026-09-01,14.00,6.35,4.0
1,2026-08-01,14.00,6.33,4.0
2,2026-07-01,14.00,5.98,4.0
3,2026-06-01,14.25,6.02,4.0
4,2026-05-01,14.50,5.31,4.0


In [46]:
file = file.set_index('date').sort_index()

In [47]:
file.head()

,key_rate,infl,target
date,,,
2013-09-01,5.5,6.14,NaN
2013-10-01,5.5,6.27,NaN
2013-11-01,5.5,6.50,NaN
2013-12-01,5.5,6.47,NaN
2014-01-01,5.5,6.07,NaN


***
<center>Interactive Visualisation<center>

In [83]:
df = (file[['key_rate', 'infl']]
      .rename(columns={'key_rate': 'Ключевая ставка', 'infl': 'Инфляция, г/г'})
      .reset_index()
      .melt('date', var_name='Ряд', value_name='Значение'))

nearest = alt.selection_point(nearest=True, on='pointerover', fields=['date'], empty=False)
brush = alt.selection_interval(encodings=['x'])   # выделение периода на мини-графике

base = alt.Chart(df).encode(
    x=alt.X('date:T', title=None, scale=alt.Scale(domain=brush, nice=False),
            axis=alt.Axis(format='%Y', tickCount='year', labelAlign='left')),
    y=alt.Y('Значение:Q', title='%'),
    color=alt.Color('Ряд:N', legend=alt.Legend(title=None, orient='top')))

lines = base.mark_line(clip=True)
points = (base.mark_circle(size=80, clip=True)
          .encode(opacity=alt.condition(nearest, alt.value(1), alt.value(0)))
          .add_params(nearest))
labels = (base.transform_calculate(
              label="format(datum['Значение'], '.2f') + '%  (' + timeFormat(datum.date, '%m.%Y') + ')'")
          .mark_text(align='left', dx=8, dy=-10, fontWeight='bold', clip=True)
          .encode(text=alt.condition(nearest, 'label:N', alt.value(''))))
target = alt.Chart(pd.DataFrame({'y': [4]})).mark_rule(strokeDash=[5, 5], color='gray').encode(y='y:Q')

main = (lines + points + labels + target).properties(
    width=850, height=400, title='Инфляция и ключевая ставка ЦБ, 2013–2026')

# мини-график снизу: на нём мышкой выделять период
overview = (alt.Chart(df).mark_line()
            .encode(x=alt.X('date:T', title=None, scale=alt.Scale(nice=False),
                            axis=alt.Axis(format='%Y', tickCount='year', labelAlign='left')),
                    y=alt.Y('Значение:Q', title=None, axis=alt.Axis(labels=False, ticks=False)),
                    color=alt.Color('Ряд:N', legend=None))
            .add_params(brush)
            .properties(width=850, height=70))
alt.renderers.set_embed_options(actions={'export': True, 'source': False,
                                         'compiled': False, 'editor': False})
main & overview

alt.VConcatChart(...)

## Выводы

1. **Три всплеска инфляции:**
   - **03.2015 — 16.9%**: падение цен на нефть, санкции и продуктовое эмбарго 2014 г.;
     рубль подешевел почти вдвое;
   - **04.2022 — 17.8%**: санкции, резкое ослабление рубля, ажиотажный спрос;
   - **03.2025 — 10.3%**: перегрев экономики - спрос выше возможностей производства,
     дефицит работников, быстрый рост зарплат и кредитования; ослабление рубля в конце 2024 г.

   Первые два всплеска — внешние шоки через курс, третий — избыточный внутренний спрос.

2. **Ставка достигает пика раньше инфляции** (на 2–5 месяцев): ЦБ реагирует на причину
   (падение рубля, перегрев), а инфляция по инерции ещё растёт и снижается уже после.
   Это согласуется с лагом денежно-кредитной политики.

3. **В 2017–2020 гг. инфляция держалась около цели 4%** (от 2.2% до ~5%):
   в спокойные периоды таргетирование работает.

4. **Ряд на вид нестационарен:** средний уровень меняется, есть разные режимы
   (спокойный около цели и шоковый). 

---
**Источники**
- Данные: [Банк России — Инфляция и ключевая ставка](https://www.cbr.ru/hd_base/infl/)
- Причины решений по ставке: [Банк России — пресс-релизы по ключевой ставке](https://www.cbr.ru/press/keypr/)
- [Банк России — Денежно-кредитная политика](https://www.cbr.ru/dkp/)

In [84]:
file.to_csv('data.csv')